# Vietnamese LPRNet OCR Training & ONNX Export

This notebook trains a lightweight, dedicated **33-Class Vietnamese License Plate Recognition Network (LPRNet)** using **11,000+ real cropped Vietnamese plates** from `lephamcong/PBL4_Deep-Learning`.

- **Target Size**: ~1.8 MB (`.onnx`)
- **Target Latency**: ~10–15 ms on CPU (95% faster than PaddleOCR)
- **Dataset**: 11,000+ real cropped Vietnamese plates (cloned directly into Colab's cloud memory—zero local disk bloat!)
- **Character Vocabulary**: Standard Vietnamese civilian alphanumeric set (`0-9`, `A-Z` excluding I, J, O, Q, W)

In [1]:
# Step 1: Environment Setup & Clone Real Vietnamese Dataset Directly to Colab Cloud
!pip install -q onnx onnxruntime
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import random
import os
import time
import glob

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Clone the 11k real cropped Vietnamese plates directly to Colab's cloud memory
# (Takes ~15 seconds, never touches your local laptop storage!)
if not os.path.exists("PBL4_Deep-Learning"):
    print("Cloning real Vietnamese plate dataset from GitHub directly to cloud...")
    !git clone --depth 1 https://github.com/lephamcong/PBL4_Deep-Learning.git
else:
    print("Dataset available in Colab environment.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 81.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.3 MB/s eta 0:00:00:00:0100:01
Using device: cuda
GPU: Tesla T4
Cloning real Vietnamese plate dataset from GitHub directly to cloud...
Cloning into 'PBL4_Deep-Learning'...
remote: Enumerating objects: 22268, done.
remote: Counting objects: 100% (22268/22268), done.
remote: Compressing objects: 100% (22126/22126), done.
remote: Total 22268 (delta 155), reused 22243 (delta 138), pack-reused 0 (from 0)
Receiving objects: 100% (22268/22268), 50.06 MiB | 22.28 MiB/s, done.
Resolving deltas: 100% (155/155), done.
Updating files: 100% (27378/27378), done.


In [9]:
# Step 2: Define Vietnamese Character Vocabulary
# Index 0 is reserved as CTC blank character '_'
# Remaining 32 chars: 10 digits + 21 Vietnamese valid plate letters + hyphen bc vetnamese plates have 32 classes.
CHARS = ['_'] + list("0123456789ABCDEFGHKLMNPRSTUVXYZ-")
CHARS_DICT = {char: i for i, char in enumerate(CHARS)}
NUM_CLASSES = len(CHARS)

print(f"Total classes (including CTC blank): {NUM_CLASSES}")
print(f"Vocabulary: {' '.join(CHARS)}")

Total classes (including CTC blank): 33
Vocabulary: _ 0 1 2 3 4 5 6 7 8 9 A B C D E F G H K L M N P R S T U V X Y Z -


In [ ]:
# Step 3: Dataset Deduplication & Synchronized Annotation Cleanup
# Scans all images, computes 64-bit difference perceptual hashes (dHash),
# and safely purges duplicate image files along with any matching annotation files (.txt, .xml, .json).

def compute_dhash(img_path, hash_size=8):
    try:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        resized = cv2.resize(img, (hash_size + 1, hash_size))
        diff = resized[:, 1:] > resized[:, :-1]
        return sum([2 ** i for (i, v) in enumerate(diff.flatten()) if v])
    except Exception:
        return None

real_data_dir = "PBL4_Deep-Learning/Dataset/BiensoxeVietNam"
all_img_paths = glob.glob(os.path.join(real_data_dir, "**", "*.jpg"), recursive=True)
all_img_paths.extend(glob.glob(os.path.join(real_data_dir, "**", "*.png"), recursive=True))

print(f"Initial scan: Found {len(all_img_paths):,} images in {real_data_dir}")

seen_hashes = {}
deleted_images = 0
deleted_annotations = 0

for p in all_img_paths:
    h = compute_dhash(p)
    if h is None:
        continue
    if h in seen_hashes:
        # Duplicate detected! Delete image and corresponding annotation file(s)
        try:
            if os.path.exists(p):
                os.remove(p)
                deleted_images += 1
            
            # Check and delete corresponding annotation files in same directory
            stem, _ = os.path.splitext(p)
            for ext in ['.txt', '.xml', '.json', '.label']:
                annot_path = stem + ext
                if os.path.exists(annot_path):
                    os.remove(annot_path)
                    deleted_annotations += 1
            
            # Also check parallel 'labels' folder if present
            dir_name = os.path.dirname(p)
            base_name = os.path.basename(stem)
            parent_dir = os.path.dirname(dir_name)
            parallel_labels = os.path.join(parent_dir, "labels")
            if os.path.exists(parallel_labels):
                for ext in ['.txt', '.xml', '.json', '.label']:
                    annot_path = os.path.join(parallel_labels, base_name + ext)
                    if os.path.exists(annot_path):
                        os.remove(annot_path)
                        deleted_annotations += 1
        except Exception as e:
            print(f"Error removing duplicate {p}: {e}")
    else:
        seen_hashes[h] = p

print(f"Deduplication complete:")
print(f"  - Purged duplicate images      : {deleted_images:,}")
print(f"  - Purged matching annotations  : {deleted_annotations:,}")
print(f"  - Unique clean images remaining: {len(seen_hashes):,}")



In [10]:
# Step 4: LPRNet Architecture Definition
class SmallBasicBlock(nn.Module):
    def __init__(self, ch_in, ch_out):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(ch_in, ch_out // 4, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch_out // 4, ch_out // 4, kernel_size=(3, 1), padding=(1, 0)),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch_out // 4, ch_out // 4, kernel_size=(1, 3), padding=(0, 1)),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch_out // 4, ch_out, kernel_size=1),
        )

    def forward(self, x):
        return self.block(x)


class LPRNet(nn.Module):
    def __init__(self, class_num=NUM_CLASSES, dropout_rate=0.5):
        super().__init__()
        self.class_num = class_num
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),  # 0
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(3, 3), stride=(1, 1)),       # 3
            SmallBasicBlock(64, 128),                               # 4
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(3, 3), stride=(2, 1)),       # 7
            SmallBasicBlock(128, 256),                              # 8
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            SmallBasicBlock(256, 256),                              # 11
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(3, 3), stride=(2, 1)),       # 14
            nn.Dropout(dropout_rate),
            nn.Conv2d(256, 256, kernel_size=(4, 1), stride=1),     # 16
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Conv2d(256, class_num, kernel_size=(1, 13), stride=1), # 20
            nn.BatchNorm2d(class_num),
            nn.ReLU(inplace=True),
        )
        self.container = nn.Sequential(
            nn.Conv2d(64 + 128 + 256 + class_num, class_num, kernel_size=1, stride=1)
        )

    def forward(self, x):
        keep_features = []
        for i, layer in enumerate(self.backbone):
            x = layer(x)
            if i in [2, 6, 13, 22]:
                keep_features.append(x)

        global_context = []
        target_size = (1, 76)
        for f in keep_features:
            if f.shape[2:] != target_size:
                f = nn.functional.interpolate(f, size=target_size, mode='bilinear', align_corners=False)
            global_context.append(f)

        merged = torch.cat(global_context, dim=1)
        logits = self.container(merged)
        return logits.squeeze(2)

model = LPRNet(class_num=NUM_CLASSES, dropout_rate=0.3).to(device)
print("Updated LPRNet model definition ready.")

Updated LPRNet model definition ready.


In [ ]:
# Step 5: Multi-Row Plate Slicing, Aspect-Ratio Normalization & Augmentation
# 1. Checks aspect ratio (w / h).
# 2. If w / h <= 1.5 (2-line plate): slices into Line 1 (top half) and Line 2 (bottom half).
# 3. Both 1-line plates and sliced 2-line halves are resized to (94, 24), matching deployment.
# 4. Applies perspective tilt, bumper margin simulation, color inversion, lighting, and blur.

def extract_plate_label(filename):
    stem = os.path.splitext(os.path.basename(filename))[0]
    label = stem.split('_')[0].upper().replace('-', '').replace('.', '')
    return label

def split_motorbike_label(full_label):
    # Vietnamese motorcycle plates: Line 1 = Province + Series, Line 2 = Serial digits
    # e.g., '29H133979' -> ('29H1', '33979')
    # e.g., '59D242069' -> ('59D2', '42069')
    # e.g., '29M6840'   -> ('29M', '6840')
    if len(full_label) >= 8 and full_label[2:4].isalnum() and not full_label[2:4].isdigit():
        return full_label[:4], full_label[4:]
    elif len(full_label) == 7 or len(full_label) == 8:
        return full_label[:3], full_label[3:]
    else:
        half = len(full_label) // 2
        return full_label[:half], full_label[half:]

class RealVietnameseDataset(Dataset):
    def __init__(self, sample_records, is_train=True):
        # sample_records: list of tuples (img_path, slice_mode, label_str)
        self.sample_records = sample_records
        self.is_train = is_train

    def __len__(self):
        return len(self.sample_records)

    def __getitem__(self, idx):
        img_path, slice_mode, label_str = self.sample_records[idx]
        img = cv2.imread(img_path)
        if img is None:
            img = np.zeros((24, 94, 3), dtype=np.uint8)
        else:
            h, w = img.shape[:2]
            # Handle multi-line plates by horizontal slicing
            if slice_mode == 'top':
                img = img[0:max(1, int(h * 0.55)), :]
            elif slice_mode == 'bottom':
                img = img[max(0, int(h * 0.45)):h, :]
            # (slice_mode == 'full' leaves img intact)

            if self.is_train:
                # 1. Random Perspective / Affine Tilt (simulates overhead camera angle)
                if random.random() < 0.6:
                    h_cur, w_cur = img.shape[:2]
                    pts1 = np.float32([[0, 0], [w_cur, 0], [0, h_cur], [w_cur, h_cur]])
                    dx = random.uniform(-0.08, 0.08) * w_cur
                    dy = random.uniform(-0.12, 0.12) * h_cur
                    pts2 = np.float32([
                        [max(0, dx), max(0, dy)],
                        [min(w_cur, w_cur - dx), max(0, -dy)],
                        [max(0, -dx), min(h_cur, h_cur + dy)],
                        [min(w_cur, w_cur + dx), min(h_cur, h_cur - dy)]
                    ])
                    matrix = cv2.getPerspectiveTransform(pts1, pts2)
                    img = cv2.warpPerspective(img, matrix, (w_cur, h_cur), borderMode=cv2.BORDER_REPLICATE)

                # 2. Random Bumper Margin Simulation (simulates YOLO boxes capturing dark bumper plastic)
                if random.random() < 0.4:
                    pad_top = random.randint(0, max(1, int(img.shape[0] * 0.12)))
                    pad_bot = random.randint(0, max(1, int(img.shape[0] * 0.12)))
                    pad_left = random.randint(0, max(1, int(img.shape[1] * 0.08)))
                    pad_right = random.randint(0, max(1, int(img.shape[1] * 0.08)))
                    bg_color = [random.randint(15, 60)] * 3
                    img = cv2.copyMakeBorder(img, pad_top, pad_bot, pad_left, pad_right, cv2.BORDER_CONSTANT, value=bg_color)

                # 3. Color Inversion / Polarity Reversal (white-on-dark vs dark-on-white)
                if random.random() < 0.15:
                    img = 255 - img

                # 4. Random Lighting & Contrast Jitter
                if random.random() < 0.5:
                    alpha = random.uniform(0.75, 1.25)
                    beta = random.randint(-25, 25)
                    img = np.clip(alpha * img.astype(np.float32) + beta, 0, 255).astype(np.uint8)

                # 5. Random Mild Blur
                if random.random() < 0.25:
                    ksize = random.choice([3, 5])
                    img = cv2.GaussianBlur(img, (ksize, ksize), 0)

            # Uniform resize to 94x24 single-row LPRNet input dimension
            img = cv2.resize(img, (94, 24))

        # Normalize to [-1.0, 1.0] matching lpr_engine.py
        norm = (img.astype(np.float32) - 127.5) * 0.0078125
        tensor = torch.tensor(norm, dtype=torch.float32).permute(2, 0, 1)

        target = [CHARS_DICT[c] for c in label_str if c in CHARS_DICT]
        if len(target) == 0:
            target = [CHARS_DICT['0']]
        return tensor, torch.tensor(target, dtype=torch.long), len(target)

def collate_fn(batch):
    images = torch.stack([item[0] for item in batch], dim=0)
    targets = [item[1] for item in batch]
    target_lengths = torch.tensor([item[2] for item in batch], dtype=torch.long)
    flat_targets = torch.cat(targets)
    return images, flat_targets, target_lengths

# Build sample records with aspect-ratio partitioning
print("Analyzing plate aspect ratios and creating single-row sample records...")
clean_files = [p for p in seen_hashes.values() if os.path.exists(p)]
random.seed(42)
random.shuffle(clean_files)

all_sample_records = []
for p in clean_files:
    full_lbl = extract_plate_label(p)
    if not full_lbl:
        continue
    # Check natural aspect ratio
    img_tmp = cv2.imread(p)
    if img_tmp is None:
        continue
    h_tmp, w_tmp = img_tmp.shape[:2]
    ar = w_tmp / float(h_tmp) if h_tmp > 0 else 2.5
    
    if ar > 1.5:
        # 1-line rectangular plate (Car)
        all_sample_records.append((p, 'full', full_lbl))
    else:
        # 2-line square plate (Motorbike) -> split into 2 single-row strips
        lbl_top, lbl_bot = split_motorbike_label(full_lbl)
        all_sample_records.append((p, 'top', lbl_top))
        all_sample_records.append((p, 'bottom', lbl_bot))

random.shuffle(all_sample_records)
split_idx = int(len(all_sample_records) * 0.90)
train_records = all_sample_records[:split_idx]
val_records = all_sample_records[split_idx:]

train_dataset = RealVietnameseDataset(train_records, is_train=True)
val_dataset = RealVietnameseDataset(val_records, is_train=False)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f"Total single-row training samples generated: {len(all_sample_records):,}")
print(f"  - Training set   : {len(train_dataset):,} samples")
print(f"  - Validation set : {len(val_dataset):,} samples")



In [12]:
# Step 6: Training Loop with CTC Loss

EPOCHS = 50 # updated ts

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

best_loss = float("inf")

print("Beginning LPRNet Training on Real Vietnamese Plates...")
start_train_t = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, targets, target_lengths in train_loader:
        imgs = imgs.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        optimizer.zero_grad()

        logits = model(imgs)
        seq_len = logits.shape[2]
        input_lengths = torch.full((imgs.shape[0],), seq_len, dtype=torch.long, device=device)

        log_probs = logits.permute(2, 0, 1).log_softmax(dim=2)
        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    avg_loss = train_loss / max(1, len(train_loader))

    # Quick validation with per-batch input_lengths
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, targets, target_lengths in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            target_lengths = target_lengths.to(device)
            logits = model(imgs)
            seq_len = logits.shape[2]
            val_input_lengths = torch.full((imgs.shape[0],), seq_len, dtype=torch.long, device=device)
            log_probs = logits.permute(2, 0, 1).log_softmax(dim=2)
            v_loss = criterion(log_probs, targets, val_input_lengths, target_lengths)
            val_loss += v_loss.item()
    avg_val = val_loss / max(1, len(val_loader))

    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Train CTC Loss: {avg_loss:.4f} | Val CTC Loss: {avg_val:.4f}")

    if avg_val < best_loss:
        best_loss = avg_val
        torch.save(model.state_dict(), "best_lprnet.pth")

total_mins = (time.time() - start_train_t) / 60.0
print(f"Training finished in {total_mins:.1f} minutes. Best Val Loss: {best_loss:.4f}")

Beginning LPRNet Training on Real Vietnamese Plates...
Epoch [01/50] | Train CTC Loss: 2.5251 | Val CTC Loss: 0.3117
Epoch [02/50] | Train CTC Loss: 0.1203 | Val CTC Loss: 0.1144
Epoch [03/50] | Train CTC Loss: 0.0432 | Val CTC Loss: 0.0355
Epoch [04/50] | Train CTC Loss: 0.0212 | Val CTC Loss: 0.0181
Epoch [05/50] | Train CTC Loss: 0.0134 | Val CTC Loss: 0.0327
Epoch [06/50] | Train CTC Loss: 0.0102 | Val CTC Loss: 0.0120
Epoch [07/50] | Train CTC Loss: 0.0043 | Val CTC Loss: 0.0045
Epoch [08/50] | Train CTC Loss: 0.0039 | Val CTC Loss: 0.0109
Epoch [09/50] | Train CTC Loss: 0.0124 | Val CTC Loss: 0.0046
Epoch [10/50] | Train CTC Loss: 0.0066 | Val CTC Loss: 0.0052
Epoch [11/50] | Train CTC Loss: 0.0029 | Val CTC Loss: 0.0029
Epoch [12/50] | Train CTC Loss: 0.0007 | Val CTC Loss: 0.0016
Epoch [13/50] | Train CTC Loss: 0.0055 | Val CTC Loss: 0.0241
Epoch [14/50] | Train CTC Loss: 0.0042 | Val CTC Loss: 0.0010
Epoch [15/50] | Train CTC Loss: 0.0020 | Val CTC Loss: 0.0053
Epoch [16/50] |

In [ ]:
# Step 7: Test CTC Greedy Decoding on a Validation Plate Sample
model.load_state_dict(torch.load("best_lprnet.pth"))
model.eval()

sample_path, sample_mode, true_label = val_records[0]
sample_img = cv2.imread(sample_path)
h_s, w_s = sample_img.shape[:2]
if sample_mode == 'top':
    sample_img = sample_img[0:max(1, int(h_s * 0.55)), :]
elif sample_mode == 'bottom':
    sample_img = sample_img[max(0, int(h_s * 0.45)):h_s, :]

sample_img = cv2.resize(sample_img, (94, 24))
test_norm = (sample_img.astype(np.float32) - 127.5) * 0.0078125
test_tensor = torch.tensor(test_norm, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(test_tensor)
    best_path = logits.argmax(dim=1)[0].cpu().numpy()

# CTC Greedy Collapse
decoded = []
prev = 0
for idx in best_path:
    if idx != 0 and idx != prev:
        decoded.append(CHARS[idx])
    prev = idx
pred_str = "".join(decoded)

print(f"Sample Path : {sample_path} (Mode: {sample_mode})")
print(f"Ground Truth: {true_label}")
print(f"Predicted   : {pred_str}")
print(f"Match       : {true_label == pred_str}")



In [14]:
# Step 8: Export directly to ONNX Format (Ready for Models_ocr/)
model = LPRNet(class_num=NUM_CLASSES, dropout_rate=0.0).cpu()
model.load_state_dict(torch.load("best_lprnet.pth", map_location="cpu"))
model.eval()

dummy_input = torch.randn(1, 3, 24, 94, dtype=torch.float32)
onnx_export_path = "lprnet.onnx"

try:
    torch.onnx.export(
        model,
        dummy_input,
        onnx_export_path,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        opset_version=12,
        dynamo=False
    )
except TypeError:
    torch.onnx.export(
        model,
        dummy_input,
        onnx_export_path,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        opset_version=12
    )

onnx_size_mb = os.path.getsize(onnx_export_path) / (1024 * 1024)
print(f"Successfully exported: {onnx_export_path} ({onnx_size_mb:.2f} MB)")

# Verify with ONNX Runtime
import onnxruntime as ort
session = ort.InferenceSession(onnx_export_path, providers=["CPUExecutionProvider"])
ort_out = session.run(None, {"input": dummy_input.numpy()})[0]
print(f"ONNX Output Shape: {ort_out.shape} (Verified!)")

# Trigger browser download to your laptop Downloads folder
from google.colab import files
files.download(onnx_export_path)

Successfully exported: lprnet.onnx (1.96 MB)
ONNX Output Shape: (1, 33, 76) (Verified!)


/tmp/ipykernel_1666/922480664.py:10: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/tmp/ipykernel_1666/1499833014.py:62: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if f.shape[2:] != target_size:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>